In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f
import pandas as pd
from pathlib import Path

# use spark session if needed
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("GenPM-eda")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

eda_data_path = Path("/workspaces/GenPM---temporary/data/shared_dir/eda_data")
raw_pm_path = eda_data_path / "raw_pm_path"
sample_path = eda_data_path / "sample"
pm_stats_path = eda_data_path / "stats"
pm_agg_path = eda_data_path / "agg"
pm_metadata = eda_data_path / "pm_metadata"

In [ ]:
def load_parquet_safe(
    spark,
    path,
    start_date=None,
    end_date=None,
    kpis=None,
    bts_ids=None,
    columns=None,
    max_rows=5_000_000,
    hard_limit_rows=10_000_000,
    sample_fraction=None,
    verbose=True
):
    """
    Safely load parquet data into Pandas with guardrails.

    Parameters:
    - spark: SparkSession
    - path: parquet path
    - start_date, end_date: date filters (string: 'YYYY-MM-DD')
    - kpis: list of kpi_id to filter
    - car_types: list of car_type to filter
    - columns: list of columns to select
    - max_rows: soft limit (will auto-limit)
    - hard_limit_rows: hard fail limit
    - sample_fraction: optional sampling (0 < x <= 1)
    - verbose: print debug info
    """

    df = spark.read.parquet(path)

    if verbose:
        df.printSchema()

    if start_date:
        df = df.filter(f.col("start_date") >= start_date)
    if end_date:
        df = df.filter(f.col("start_date") <= end_date)

    if kpis:
        df = df.filter(f.col("kpi_id").isin(kpis))

    if bts_ids:
        df = df.filter(f.col("car_type").isin(bts_ids))

    if columns:
        df = df.select(columns)

    if sample_fraction:
        df = df.sample(fraction=sample_fraction, seed=42)

    approx_count = df.rdd.isEmpty() and 0 or df.limit(hard_limit_rows).count()

    if verbose:
        print(f"[INFO] Approx row count: {approx_count}")

    if approx_count > hard_limit_rows:
        raise ValueError(
            f"Dataset too large ({approx_count} rows). "
            f"Refine filters or use sampling."
        )

    if approx_count > max_rows:
        if verbose:
            print(f"[WARN] Limiting rows to {max_rows}")
        df = df.limit(max_rows)

    pdf = df.orderby("start_time").toPandas()

    if verbose:
        print(f"[INFO] Loaded {len(pdf)} rows into Pandas")

    return pdf

# EDA - template
#### Author: --

In [ ]:
simple_reports_df = pd.read_parquet(str(pm_metadata / "simple_reports"))

In [ ]:
simple_reports_df.info()

In [ ]:
pm_raw_df = spark.read.parquet(str(eda_data_path / "raw_pm_data"))
# test
pm_raw_df.printSchema()
print(pm_raw_df.count())

In [ ]:
pm_test_df = load_parquet_safe(spark, str(eda_data_path / "raw_pm_data"))

In [ ]:
pm_test_df